# Pizza Cluster - Full Orchestration Pipeline

Questo notebook è una dashboard operativa (Orchestrator) che collega l'intero ciclo di vita dei dati per il progetto **Pizza Cluster**:
1. **Data Extraction**: Scaricamento del dataset JMAIL (Parquet).
2. **Data Processing**: Preprocessing testuale, pulizia conservativa e generazione di feature numeriche e temporali.
3. **Embeddings**: Generazione dei vector embeddings tramite `BAAI/bge-small-en-v1.5`.
4. **Clustering**: Riduzione dimensionale (UMAP) e clustering (HDBSCAN) sui vettori semantici.
5. **Topic Labeling & Naming**: Estrazione delle parole chiave (c-TF-IDF) e inferenza di un nome riassuntivo (LLM via Ollama).

> **Nota**: Il notebook richiama direttamente i moduli standardizzati dalla cartella `src/utils/` secondo le linee guida architetturali.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Aggiungiamo la root del progetto al path per permettere l'import da src
project_root = os.path.abspath(os.path.join('..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Importiamo le utility modulari del progetto
from src.utils.data_processing import run_processing, run_processing_with_limit
from src.utils.embedding_pipeline import run_embedding_pipeline
from src.utils.clustering_hdbscan import run_umap_hdbscan
from src.utils.topic_labeling import calculate_ctfidf
from src.utils.llm_naming import get_llm_cluster_name

env_path = os.path.join(project_root, '.env')


## 1. Data Preprocessing

La pipeline esegue le seguenti trasformazioni come definito in `DATA_CONTRACTS.md`:
- Filtra le email promozionali.
- Crea il campo `combined_text` (oggetto + corpo dell'email).
- Aggiunge feature diagnostiche: contatore redazioni (`[redacted]`), recipient counts, word counts.

*Scegli se eseguire il preprocessing su un sample limitato o sull'intero dataset.*

In [ ]:
# LIMIT_ROWS = 1000 # Rimuovere o impostare a None per elaborare tutto il file
LIMIT_ROWS = 5000 

print(f"Avvio il preprocessing (limite: {LIMIT_ROWS} righe)...")
if LIMIT_ROWS:
    processing_result = run_processing_with_limit(env_file=env_path, limit=LIMIT_ROWS)
else:
    processing_result = run_processing(env_file=env_path)

print("\nRisultato del preprocessing:")
for key, value in processing_result.items():
    print(f"  {key}: {value}")

processed_parquet_path = processing_result.get('processed_output_path', os.path.join(project_root, 'data', 'processed', 'jmail_emails_processed.parquet'))
df_processed = pd.read_parquet(processed_parquet_path)
print(f"\nDataset processato caricato in memoria. Dimensioni: {df_processed.shape}")
df_processed.head(2)


## 2. Generazione Embeddings

Generiamo gli embeddings per il `combined_text` utilizzando il modello `BAAI/bge-small-en-v1.5`, come approvato nel registro decisioni (`DECISIONS.md`).

In [ ]:
print("Avvio generazione embeddings...")
# La pipeline leggerà in automatico l'input configurato (ad esempio il file elaborato in output dal blocco precedente)
embedding_result = run_embedding_pipeline(env_file=env_path)

print("\nRisultato generazione embeddings:")
for key, value in embedding_result.items():
    print(f"  {key}: {value}")

embeddings_path = embedding_result.get('embedding_vectors_path', os.path.join(project_root, 'data', 'embeddings', 'email_embeddings.npy'))
embeddings = np.load(embeddings_path)
print(f"\nEmbeddings caricati in memoria. Forma: {embeddings.shape}")


## 3. Clustering (UMAP + HDBSCAN)

I vettori a 384 dimensioni vengono ridotti a 15 dimensioni tramite UMAP per risolvere la *maledizione della dimensionalità*.
Successivamente viene applicato HDBSCAN utilizzando i parametri ottimizzati trovati via Grid Search:
- `min_cluster_size` = 200 (da abbassare se si usa un piccolo sample)
- `min_samples` = 10

In [ ]:
print("Avvio UMAP + HDBSCAN...")

# Parametri approvati per l'intero dataset:
min_cluster_size = 200
min_samples = 10

# Se stiamo processando un sample, abbassiamo artificialmente la soglia
if len(embeddings) < 10000:
    print("ATTENZIONE: Dataset di dimensioni ridotte rilevato, uso parametri di clustering per sample.")
    min_cluster_size = max(15, len(embeddings) // 100)
    min_samples = max(5, min_cluster_size // 4)

hdbscan_labels, hdbscan_probs, reduced_embeddings = run_umap_hdbscan(
    embeddings=embeddings,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples
)

df_processed['cluster'] = hdbscan_labels
df_processed['cluster_prob'] = hdbscan_probs

n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
n_noise = list(hdbscan_labels).count(-1)

print(f"\nRisultato Clustering:")
print(f" - Cluster identificati: {n_clusters}")
print(f" - Documenti rumorosi (Outlier): {n_noise} ({n_noise / len(hdbscan_labels) * 100:.2f}%)")


## 4. Topic Labeling e LLM Naming

Estrazione semantica:
1. Raggruppiamo il testo (`combined_text`) per ogni cluster.
2. Calcoliamo le **keyword** principali usando `c-TF-IDF`.
3. Inviamo le keyword a un **LLM Locale (Ollama)** tramite Few-Shot Prompt per riassumerle in un nome descrittivo di 1-3 parole.

In [ ]:
print("Preparazione dati per c-TF-IDF...")
cluster_docs = {}

# Raggruppiamo i documenti escludendo gli outlier (-1)
for cluster_id, group in df_processed[df_processed['cluster'] != -1].groupby('cluster'):
    # Limitiamo il numero di testi concatenati per questioni di memoria
    docs = group['combined_text'].fillna("").tolist()
    cluster_docs[cluster_id] = " ".join(docs[:100])

print(f"Calcolo c-TF-IDF e keyword extraction per {len(cluster_docs)} cluster...")
# Calcoliamo le prime 15 keyword
cluster_keywords = calculate_ctfidf(cluster_docs, top_n=15)

# Prepariamo un dizionario per i nomi
cluster_names = {}
print("\nGenerazione nomi cluster via LLM locale (Ollama)...")
print("Questo passaggio può richiedere alcuni minuti a seconda dell'hardware.\n")

# Per la demo, analizziamo solo i primi 10 cluster
max_clusters_to_name = 10
clusters_to_process = list(cluster_keywords.items())[:max_clusters_to_name]

for cluster_id, keywords in clusters_to_process:
    kw_str = ", ".join(keywords)
    print(f"Cluster {cluster_id} | Keywords: {kw_str}")
    try:
        # Usa Qwen 3.5 o Llama3 (a seconda della disponibilità locale)
        # Nota: L'utente nel context menzionava llama3 e Qwen 3.5. Default a llama3.
        llm_name = get_llm_cluster_name(keywords, model="llama3")
        cluster_names[cluster_id] = llm_name
        print(f"  -> Nome Generato: {llm_name}\n")
    except Exception as e:
        print(f"  -> ERRORE durante chiamata Ollama: {e}\n")
        cluster_names[cluster_id] = "Error/Unknown"



## 5. Analisi e Salvataggio Risultati

Visualizziamo il DataFrame riassuntivo dei cluster e salviamo le label per integrazioni future.

In [ ]:
# Summary finale
summary_data = []
for cluster_id in cluster_names.keys():
    size = len(df_processed[df_processed['cluster'] == cluster_id])
    summary_data.append({
        'Cluster ID': cluster_id,
        'Dimensione': size,
        'LLM Name': cluster_names[cluster_id],
        'Keywords (c-TF-IDF)': ", ".join(cluster_keywords[cluster_id][:7])
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)

# Opzionale: Salvataggio dataframe annotato
# df_processed.to_parquet(os.path.join(project_root, 'data', 'processed', 'jmail_emails_clustered.parquet'), index=False)
